# 第 2 章：MDP 与贝尔曼方程 —— RL 的数学语言

> 这一章是整本书的**地基**。把贝尔曼方程吃透，后面所有算法（DQN、PPO、GRPO）都能看懂。

## 学习目标

1. 理解 **马尔可夫决策过程（MDP）** 的五元组 $(S, A, P, R, \gamma)$
2. 掌握 **状态价值 $V^\pi$** 和 **动作价值 $Q^\pi$** 的定义
3. **逐步推导贝尔曼期望方程**（最重要！）
4. 通过交互式 widget 直观感受 $\gamma$ 对 $V$ 的影响
5. 看"值传播"动画：奖励如何从终点倒着传回起点

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'rlenvs').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from utils import set_seed, plot_value_heatmap, make_interactive
from rlenvs import GridWorld, small_grid_5x5, bridge_grid

set_seed(0)

## 2.1 从 bandit 到 MDP：引入"状态"

第 1 章的老虎机已经能平衡探索与利用了，但它有一个被我们悄悄绕过的假设：**今天拉哪根拉杆，不会改变明天有哪些拉杆**。每次面对的都是同一个问题、同一组选择。

下围棋可就不是这样了——这一步落在哪里，直接决定了棋盘接下来长什么样、你还有哪些选择。一旦**动作开始改变世界**，"眼前这一步拿多少奖励"就远远不够了：一个贪吃子的棋手，每步的即时奖励都很高，然后在二十手后被围死。

所以我们需要一种新的语言，能够回答两个 bandit 回答不了的问题：

1. **状态怎么转移？** 选了动作 $a$ 之后，世界会变成什么样？
2. **奖励怎么和长期挂钩？** 当前一步的甜头和未来的收益，如何权衡？

这就是 MDP。在动形式化定义之前，先在三个熟悉场景里找找"状态"的影子：

- 围棋：当前棋盘就是状态，不同棋盘对应不同最优动作
- 开车：位置 + 速度 + 路况构成状态
- LLM：当前已生成的 token 序列就是状态

### MDP 五元组

一个**马尔可夫决策过程**由五元组 $\mathcal{M} = (\mathcal{S}, \mathcal{A}, P, R, \gamma)$ 刻画：

| 符号 | 含义 |
|---|---|
| $\mathcal{S}$ | 状态集合（state space） |
| $\mathcal{A}$ | 动作集合 |
| $P(s' \| s, a)$ | **转移概率**：在 $s$ 选 $a$，到 $s'$ 的概率 |
| $R(s, a)$（或 $R(s'a s)$） | 在 $s$ 选 $a$（再到 $s'$）的期望奖励 |
| $\gamma \in [0, 1]$ | **折扣因子** |

### 马尔可夫性

**下一个状态和奖励只依赖当前状态和动作，不依赖更早的历史**：

$$
P(s_{t+1} = s' \mid s_t, a_t, s_{t-1}, a_{t-1}, \dots) = P(s_{t+1} = s' \mid s_t, a_t)
$$

这是个**假设**——很多现实问题严格说不满足马尔可夫性（比如打牌时弃牌堆的信息），但**通常能找到一个充分包含信息的 state 表征**让马尔可夫性近似成立。LLM 中"上下文窗口"就是为近似马尔可夫性服务的。

## 2.2 回报与折扣：为什么要 $\gamma$

agent 的目标是最大化 **期望累计奖励**：

$$
G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}
$$

### 为什么必须 $\gamma < 1$？

三个理由：

1. **数学**：避免无穷大。如果 episode 无限长、$\gamma = 1$，则 $G_t$ 可能发散
2. **人类直觉**：今天的 1 元 > 明天的 1 元（货币的时间价值）
3. **算法收敛**： discounted MDP 一定有有限的最优值（Ch03 会用到）

### $\gamma$ 的现实意义

- $\gamma = 0$：只看眼前奖励（**目光短浅**），$V^\pi(s) = \mathbb{E}[R_{t+1}]$
- $\gamma = 1$：未来奖励和眼前一样重要（**无限远见**），数学上危险
- $\gamma \in [0.9, 0.99]$：典型取值。**越大越难学，但策略越优**

### $G_t$ 的自递推（递归形式）

**关键观察**：回报可以递归定义。

$$
G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = R_{t+1} + \gamma \big[ R_{t+2} + \gamma R_{t+3} + \dots \big] = R_{t+1} + \gamma G_{t+1}
$$

这个简单的等式 $G_t = R_{t+1} + \gamma G_{t+1}$ 几乎是所有 RL 算法的根源。

## 2.3 策略、状态价值、动作价值

### 策略 $\pi(a|s)$

策略是从状态到动作分布的映射：$\pi(a|s) = \Pr[A_t = a \mid S_t = s]$。

它可以是确定性的（$\pi(s)$ 总返回同一动作），也可以是随机的。

### 状态价值函数 $V^\pi(s)$

**在状态 $s$ 出发、之后按策略 $\pi$ 行动，期望累计奖励**：

$$
V^\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]
$$

### 动作价值函数 $Q^\pi(s, a)$

**在状态 $s$ 选 $a$、之后按 $\pi$ 行动，期望累计奖励**：

$$
Q^\pi(s, a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a]
$$

**两者关系**：

$$
V^\pi(s) = \sum_a \pi(a|s) Q^\pi(s, a)
$$

即 $V$ 是 $Q$ 在策略下的期望。

## 2.4 贝尔曼期望方程（**最重要！**）

现在手上有了 $V^\pi$ 的定义，但按定义算它要把**所有可能的未来**都加一遍——围棋有 $10^{170}$ 种未来，一次都枚举不动。怎么办？

> 🤔 **先自己猜 30 秒**：$V^\pi(s)$（在 $s$ 出发的期望总回报）和 $V^\pi(s')$（在下一个状态出发的期望总回报）之间，应该存在什么关系？
>
> <details><summary>想好了点开对照</summary>
>
> 直觉：「一个状态的价值 = 马上能拿的奖励 + 打了折之后的下一个状态的价值」。
> 如果这个递归成立，我们就不需要枚举整棵未来树——每个状态只跟它的**后继**有关。
> 把这个直觉写成等式并严格证明它，就是本节的内容。
> </details>

我们想把 $V^\pi$ 和 $Q^\pi$ 写成递归形式——这就是**贝尔曼方程**。

### 直觉

$V^\pi(s)$ 是从 $s$ 出发的期望回报。它等于：

> 即时奖励 + $\gamma$ × 下一状态的价值（期望）

### 推导 $V^\pi$ 的贝尔曼方程

$$
\begin{aligned}
V^\pi(s) &= \mathbb{E}_\pi[G_t \mid S_t = s] \\
         &= \mathbb{E}_\pi[R_{t+1} + \gamma G_{t+1} \mid S_t = s]  \quad (\text{用 } G_t = R_{t+1} + \gamma G_{t+1}) \\
         &= \mathbb{E}_\pi[R_{t+1}] + \gamma \, \mathbb{E}_\pi[G_{t+1} \mid S_t = s]
\end{aligned}
$$

**第一项**：$\mathbb{E}_\pi[R_{t+1} \mid S_t = s]$。需要先选 $a$（概率 $\pi(a|s)$），然后从 $(s, a)$ 采样的奖励期望是 $R(s, a)$：

$$
\mathbb{E}_\pi[R_{t+1} \mid S_t = s] = \sum_a \pi(a|s) R(s, a) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) r(s, a, s')
$$

（最后一项在 $r$ 与 $s'$ 也有关时用）

**第二项**：$\mathbb{E}_\pi[G_{t+1} \mid S_t = s]$。需要选 $a$（$\pi$），转移到 $s'$（$P$），然后从 $s'$ 出发的期望回报是 $V^\pi(s')$：

$$
\mathbb{E}_\pi[G_{t+1} \mid S_t = s] = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) V^\pi(s')
$$

合起来：

$$
\boxed{\; V^\pi(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) \big[ r(s,a,s') + \gamma V^\pi(s') \big] \;}
$$

这就是**贝尔曼期望方程**。"期望"是因为我们对**策略 $\pi$** 和**转移 $P$** 都取期望。

### 推导 $Q^\pi$ 的贝尔曼方程

类似地：

$$
\boxed{\; Q^\pi(s, a) = \sum_{s'} P(s'|s,a) \big[ r(s,a,s') + \gamma \sum_{a'} \pi(a'|s') Q^\pi(s', a') \big] \;}
$$

或者用 $V$ 表达：

$$
Q^\pi(s, a) = \sum_{s'} P(s'|s,a) \big[ r(s,a,s') + \gamma V^\pi(s') \big]
$$

### 矩阵形式（为 Ch03 做铺垫）

若有 $n$ 个状态，$V^\pi$ 是 $n$ 维向量，则：

$$
V^\pi = R^\pi + \gamma P^\pi V^\pi
$$

其中 $P^\pi_{ss'} = \sum_a \pi(a|s) P(s'|s,a)$、$R^\pi_s = \sum_a \pi(a|s) R(s,a)$。

这是一个**线性方程组**！Ch03 我们会看到怎么解。

## 2.5 看一眼真实 MDP：small_grid_5x5

我们用一个 $5 \times 5$ 的网格：
- 起点 $(4, 0)$（左下角）
- 终点 $(0, 4)$（右上角，奖励 $+1$）
- 一个陷阱 $(1, 2)$（奖励 $-0.5$）
- 两堵墙 $(2, 2), (2, 3)$
- 每步默认 $-0.05$（让 agent 别磨蹭）

我们环境**完全暴露** $P[s, a, s']$ 和 $R[s, a]$，可以直接用它们算 $V^\pi$。

In [ ]:
env = small_grid_5x5(seed=0)
print(f"shape: {env.shape}, n_states={env.nS}, n_actions={env.nA}")
print(f"动作: 0=↑, 1=→, 2=↓, 3=←")
print(f"终点: {env.terminals}")
print(f"墙:   {env.walls}")
print(f"特殊奖励: {env.rewards}")
print()
print("P[s, a, s'] 的形状：", env.P.shape)
print("R[s, a] 的形状：    ", env.R.shape)
print()
# 验证 P 行和为 1
print("P 每行和：", env.P.sum(axis=2).min(), "~", env.P.sum(axis=2).max())

## 2.6 手动计算均匀随机策略的 $V^\pi$

我们让策略 $\pi(a|s) = 1/4$ 对每个动作（均匀随机）。

### 用矩阵形式解 $V^\pi$

$V^\pi = R^\pi + \gamma P^\pi V^\pi \Rightarrow (I - \gamma P^\pi) V^\pi = R^\pi \Rightarrow V^\pi = (I - \gamma P^\pi)^{-1} R^\pi$

这是**精确解**，不用迭代。

In [ ]:
def compute_uniform_random_V(env, gamma=0.9):
    """用矩阵求逆精确解 V^π，π 是均匀随机策略。"""
    nS, nA = env.nS, env.nA
    # 构造 π(a|s) = 1/nA 的矩阵 [nS, nA]
    pi = np.full((nS, nA), 1.0 / nA)
    # R^π[s] = Σ_a π(a|s) R(s, a)
    R_pi = (pi * env.R).sum(axis=1)
    # P^π[s, s'] = Σ_a π(a|s) P(s'|s, a)
    P_pi = np.einsum('sa,saq->sq', pi, env.P)
    # 解线性方程组
    A = np.eye(nS) - gamma * P_pi
    V = np.linalg.solve(A, R_pi)
    return V, R_pi, P_pi


V, R_pi, P_pi = compute_uniform_random_V(env, gamma=0.9)
print("均匀随机策略下的 V^π（精确解）：")
print(V.reshape(env.shape).round(2))
print()

# 画热力图
fig, ax = plt.subplots(figsize=(5, 5))
plot_value_heatmap(
    V, env.shape, cell_text=True,
    walls=list(env.walls), terminals=list(env.terminals),
    ax=ax, title='V^π (random policy, γ=0.9)',
)
plt.tight_layout(); plt.show()

注意几个现象：

1. **终点 (0, 4) 处 V=0**：终止态不再产生奖励（这一步没拿奖励）
2. **越靠近终点 V 越高**：因为可以更快到达 +1
3. **陷阱 (1, 2) 处 V 是负数**：踩上去 -0.5
4. **墙没价值**：不可达

### 数值验证：蒙特卡洛估计

矩阵法得到的 $V$ 对不对？我们用蒙特卡洛验证一下：从状态 $s$ 出发、按均匀随机策略走，直到终点，记录累计奖励的均值。

In [ ]:
def uniform_random_policy(s):
    return np.random.randint(4)

def mc_estimate_V(env, gamma=0.9, n_episodes=2000, max_steps=200):
    """用 MC 估计每个非终止、非墙状态的 V^π。"""
    V_sum = np.zeros(env.nS)
    V_cnt = np.zeros(env.nS)
    for s0 in range(env.nS):
        # 跳过终止态和墙
        if env.is_terminal(s0):
            continue
        if env.state_to_xy(s0) in env.walls:
            continue
        for _ in range(n_episodes):
            env._state = s0
            G = 0.0
            t = 0
            done = False
            while not done and t < max_steps:
                a = uniform_random_policy(env.state)
                _, r, done, _ = env.step(a)
                G += (gamma ** t) * r
                t += 1
            V_sum[s0] += G
            V_cnt[s0] += 1
    V_mc = np.where(V_cnt > 0, V_sum / np.maximum(V_cnt, 1), 0.0)
    return V_mc, V_cnt


V_mc, V_cnt = mc_estimate_V(env, gamma=0.9, n_episodes=2000, max_steps=200)

# 比较（只看有效状态）
print(f"{'state':<6}{'V_exact':<12}{'V_mc':<12}{'diff':<10}")
for s in [0, 5, 10, 15, 20, 24]:
    if V_cnt[s] > 0:
        print(f"{s:<6}{V[s]:<12.3f}{V_mc[s]:<12.3f}{abs(V[s]-V_mc[s]):<10.3f}")

# 只统计有效状态的最大误差
mask = V_cnt > 0
err = np.abs(V - V_mc)[mask].max()
print(f"\n有效状态的最大绝对误差: {err:.4f}（应在 0.05 以内）")

### 数值验证的小问题

如果你看到误差比较大，可能是：
1. MC 方差大（2000 episodes 不够）→ 增加 `n_episodes`
2. 步数限制截断（200 不够，γ=0.9 时尾巴贡献有限）

后面 Ch04 我们会用 TD(0) 给出更高效的估计方法。

## 2.7 交互式 widget：$\gamma$ 如何影响 $V$

In [ ]:
def plot_V_gamma(gamma=0.9):
    V, _, _ = compute_uniform_random_V(env, gamma=gamma)
    fig, ax = plt.subplots(figsize=(5, 5))
    plot_value_heatmap(
        V, env.shape, cell_text=True,
        walls=list(env.walls), terminals=list(env.terminals),
        ax=ax, title=f'V^π (random policy, γ={gamma:.2f})',
    )
    plt.tight_layout(); plt.show()

w = make_interactive(plot_V_gamma,
                     params={'gamma': (0.9, 0.0, 0.99, 0.01)})

拖动 $\gamma$ 从 0 到 0.99，观察：

- $\gamma = 0$：所有非终点状态 V ≈ 即时奖励（多为 -0.05 或 -0.5）
- $\gamma$ 增大：值"传播"更远，远离终点的格子也开始变正
- $\gamma \to 1$：值普遍变大（因为终点 +1 的影响传得很远）

### 这个 widget 的本质

**$\gamma$ 控制 agent 的"远见程度"**。这正是 RL 中调 $\gamma$ 的核心权衡：

- $\gamma$ 小：agent 短视，只看下一步（容易学，但策略可能次优）
- $\gamma$ 大：agent 远见，看长期（更难学，但策略更优）

## 2.8 值传播动画：奖励如何"逆向"流回起点

我们做这样一个实验：
1. 初始化 $V_0(s) = 0$ 对所有 $s$
2. 迭代 $V_{k+1}(s) = R^\pi(s) + \gamma P^\pi V_k(s)$
3. 看 $V_k$ 怎么随 $k$ 变化

这正是 Ch03 的**迭代策略评估** 的核心。这里我们提前展示，让你看"值是怎么传播的"。

In [ ]:
from utils import animate_agent

def value_propagation_animation(env, gamma=0.9, n_iters=30, fps=2):
    """迭代策略评估，每一帧是一次 sweep 的 V_k。"""
    nS, nA = env.nS, env.nA
    pi = np.full((nS, nA), 1.0 / nA)
    R_pi = (pi * env.R).sum(axis=1)
    P_pi = np.einsum('sa,saq->sq', pi, env.P)
    V = np.zeros(nS)
    Vs = [V.copy()]
    for _ in range(n_iters):
        V = R_pi + gamma * P_pi @ V
        Vs.append(V.copy())

    fig, ax = plt.subplots(figsize=(5, 5))
    def update(k):
        ax.clear()
        plot_value_heatmap(
            Vs[k], env.shape, cell_text=True,
            walls=list(env.walls), terminals=list(env.terminals),
            ax=ax, title=f'sweep {k}',
        )
    anim = animation.FuncAnimation(fig, update, frames=len(Vs), interval=1000//fps, blit=False, repeat=True)
    plt.close(fig)
    return anim

anim = value_propagation_animation(env, gamma=0.9, n_iters=25, fps=2)
HTML(anim.to_jshtml())  # 在 notebook 内显示

你应该看到：

- **Sweep 0**：所有 $V = 0$
- **Sweep 1**：只有终点的邻居获得了非零值（"信号到达了邻居"）
- **Sweep 2, 3, ...**：值像水波一样传到更远的格子
- **Sweep ~20**：基本收敛到精确 $V^\pi$

**这就是"值传播"——奖励信号沿 $P$ 反向传播的过程**。理解了它，你就理解了 RL 中"学习"的本质。

## 2.9 📝 练习

### 练习 1：Bridge Grid 中 $\gamma$ 如何翻转策略

`bridge_grid()` 是一个 $3 \times 5$ 网格：
- 起点 $(1, 0)$，终点 $(1, 4)$
- 中间 $(1, 1..3)$ 是"桥"，桥下 $(2, 1..3)$ 是深渊（$-1$）
- 桥上方 $(0, *)$ 也是通路，绕远但安全

**问题**：找一个 $\gamma^*$ 阈值，使得：
- $\gamma < \gamma^*$：最优策略"绕远"（走 row 0）
- $\gamma > \gamma^*$：最优策略"抄近道"（走桥）

**提示**：
1. 对不同 $\gamma$，用矩阵法算最优 $V^*$
2. 找出"桥起点" $(1, 0)$ 和"绕远起点" $(0, 0)$ 的 $V$ 何时翻转

> 参考答案：`solutions/ch02_bridge_gamma.ipynb`

---

## 2.10 小结

| 概念 | 一句话 |
|---|---|
| MDP | $(S, A, P, R, \gamma)$ 五元组——RL 世界的完整数学描述 |
| 回报 $G_t$ | 从 $t$ 起的折扣累计奖励 $\sum_k \gamma^k r_{t+k+1}$ |
| 贝尔曼期望方程 | $V^\pi(s) = \mathbb{E}[r + \gamma V^\pi(s')]$——一切 RL 算法的根源 |
| 矩阵解 | $V = (I - \gamma P_\pi)^{-1} r_\pi$，模型已知时的精确解 |
| $Q^\pi$ | 多一个动作维度的价值；$\arg\max_a Q^* = \pi^*$ |

三个带走的东西：

1. **γ 不只是工程参数，是数学必需**：它让无限长轨迹的回报收敛，也编码了"未来多重要"
2. **贝尔曼方程是递归**：当前的值 = 即时奖励 + 折扣 × 后继的值——后面每一章都在用不同方式解这一个方程
3. **解析解 vs MC 验证**：两者在容差内一致，建立了"采样可信"的信心——这是后面所有无模型方法的地基

> 📖 学完本章，先做 `STUDY_GUIDE.md` 里 Ch02 的自测题（4 题），全对再进下一章。

---

下一章：**第 3 章 — 动态规划**。
我们将用本章的贝尔曼方程，去**精确求解 MDP**——当你"知道一切"（即知道 $P$ 和 $R$）时。